In [3]:
from ollama import chat
import json
from typing import List, Optional
from pydantic import BaseModel, Field, ValidationError, ConfigDict

image_path = "../data/image-13.png"

SCHEMA_COLUMNS = ["Prod date", "Exp date", "Product Name", "Batch No.", "pH"]

class ProductRow(BaseModel):
    model_config = ConfigDict(populate_by_name=True, extra="ignore")

    prod_date: Optional[str] = Field(default=None, alias="Prod date")
    exp_date: Optional[str] = Field(default=None, alias="Exp date")
    product_name: Optional[str] = Field(default=None, alias="Product Name")
    batch_no: Optional[str] = Field(default=None, alias="Batch No.")
    ph: Optional[str] = Field(default=None, alias="pH")

class ProductTable(BaseModel):
    model_config = ConfigDict(extra="ignore")
    rows: List[ProductRow]

def extract_json_candidate(text: str) -> str:
    """Extract a JSON-looking substring from model output."""
    stripped = text.strip()
    if stripped.startswith("{"):
        return stripped

    start = stripped.find("{")
    if start == -1:
        return stripped

    return stripped[start:]

def repair_json_text(text: str) -> str:
    """Try simple JSON repair for common LLM truncation issues."""
    candidate = extract_json_candidate(text).strip()
    if not candidate:
        return candidate

    open_curly = candidate.count("{")
    close_curly = candidate.count("}")
    if close_curly < open_curly:
        candidate = candidate + ("}" * (open_curly - close_curly))

    open_square = candidate.count("[")
    close_square = candidate.count("]")
    if close_square < open_square:
        candidate = candidate + ("]" * (open_square - close_square))

    return candidate

def coerce_table_with_pydantic(structured_text: str) -> ProductTable:
    """Validate and coerce LLM output into strict schema using Pydantic."""
    attempts = [structured_text, repair_json_text(structured_text)]

    for attempt in attempts:
        try:
            parsed_json = json.loads(extract_json_candidate(attempt))
            return ProductTable.model_validate(parsed_json)
        except (json.JSONDecodeError, ValidationError):
            continue

    fallback_response = chat(
        model="llama3.2:1b",
        messages=[
            {
                "role": "system",
                "content": (
                    "Return ONLY valid JSON with exact schema: "
                    "{\"rows\": [{\"Prod date\": null, \"Exp date\": null, \"Product Name\": null, \"Batch No.\": null, \"pH\": null}]}. "
                    "No markdown, no explanation, no extra keys."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Fix this malformed JSON and return valid JSON only:\n\n"
                    f"{structured_text}"
                ),
            },
        ],
    )

    fixed_text = fallback_response.message.content
    parsed_fixed = json.loads(repair_json_text(fixed_text))
    return ProductTable.model_validate(parsed_fixed)

def row_to_schema_dict(row: ProductRow) -> dict:
    dumped = row.model_dump(by_alias=True)
    return {
        "Prod date": dumped.get("Prod date"),
        "Exp date": dumped.get("Exp date"),
        "Product Name": dumped.get("Product Name"),
        "Batch No.": dumped.get("Batch No."),
        "pH": dumped.get("pH"),
    }

# Step 1: OCR extraction from the image
ocr_response = chat(
    model="deepseek-ocr:3b",
    messages=[
        {
            "role": "user",
            "content": "Extract all table text from this image exactly as it appears.",
            "images": [image_path],
        }
    ],
)

raw_text = ocr_response.message.content
print("=== OCR Raw Output ===")
print(raw_text)

# Step 2: Convert OCR text into strict schema rows
llm_response = chat(
    model="llama3.2:1b",
    messages=[
        {
            "role": "system",
            "content": (
                "You extract structured table rows from OCR text. "
                "Return ONLY valid JSON (no markdown, no explanation). "
                "Use this exact schema with exact keys and same case: "
                "{\"rows\": [{\"Prod date\": null, \"Exp date\": null, \"Product Name\": null, \"Batch No.\": null, \"pH\": null}]}. "
                "If multiple table rows exist, include all rows. "
                "If a field is missing or unclear, set it to null."
            ),
        },
        {
            "role": "user",
            "content": f"OCR text:\n\n{raw_text}",
        },
    ],
)

structured_text = llm_response.message.content
print("\n=== Structured JSON Output (LLM Raw) ===")
print(structured_text)

# Step 3: Pydantic validation + fallback repair
table = coerce_table_with_pydantic(structured_text)
normalized_rows = [row_to_schema_dict(r) for r in table.rows]

print("\n=== Final Structured Rows ===")
print(json.dumps({"rows": normalized_rows}, indent=2, ensure_ascii=True))

=== OCR Raw Output ===
<table><th colspan="1" rowspan="2">Prod date</th><th colspan="1" rowspan="2">Exp date</th><th colspan="1" rowspan="2">Product Name</th><th colspan="3"></th>
<td colspan="1"><b>Batch No.</b><td colspan="1"><td colspan="1">
<td colspan="1">04.11.23<td colspan="1">04.11.24<td colspan="1">DB2 Stabiliser<td colspan="1">893<td colspan="1">7.17<td colspan="1" rowspan="5">pH 1% (1g/99ml) Specification
<td colspan="1">04.11.23<td colspan="1">04.11.24<td colspan="1">DB2 Stabiliser<td colspan="1">894<td colspan="1">7.17
<td colspan="1">04.11.23<td colspan="1">04.11.24<td colspan="1">DB2 Stabiliser<td colspan="1">895<td colspan="1">7.16
<td colspan="1">04.11.23<td colspan="1">04.11.24<td colspan="1">DB2 Stabiliser<td colspan="1">896<td colspan="1">7.17
<td colspan="1">04.11.23<td colspan="1">04.11.24<td colspan="1">DB2 Stabiliser<td colspan="1">897<td colspan="1">7.13
</table>

=== Structured JSON Output (LLM Raw) ===
{"rows": [{"Prod date": "2023-11-04", "Exp date": "2023-1